In [1]:
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import average_precision_score

import pandas as pd
import numpy as np
import shutil

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
### Directories
project_root = Path.cwd()

ground_truth_directory = project_root / Path("eval/gt")
predicted_directory = project_root / Path("eval/pred")
result_directory = project_root / Path("eval/result")

ground_truth_directory.mkdir(parents=True, exist_ok=True)
predicted_directory.mkdir(parents=True, exist_ok=True)
result_directory.mkdir(parents=True, exist_ok=True)

print(f"Current Working Directory: {project_root}")

Current Working Directory: c:\Gabriel_Files\Programming_Files\School\Thesis\src


In [ ]:
### Clear Existing Output Directories
for item in result_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [ ]:
def evaluate_tiou(truth_csv_path, predicted_csv_path):
    import pandas as pd
    import numpy as np

    truth_df = pd.read_csv(truth_csv_path)
    pred_df = pd.read_csv(predicted_csv_path)
    matches = []

    for (hum_id, obj_id), truth_group in truth_df.groupby(['human_id', 'object_id']):
        pred_group = pred_df[(pred_df['human_id'] == hum_id) & (pred_df['object_id'] == obj_id)]
        
        if pred_group.empty:
            for t_idx, t_row in truth_group.iterrows():
                t_start = t_row['frame_start']
                t_end = t_row['frame_end']
                matches.append((hum_id, obj_id, t_idx, [], f"{t_start} - {t_end}", [], 0, (t_end - t_start) + 1, 0.0))
            continue
            
        for t_idx, t_row in truth_group.iterrows():
            t_start = t_row['frame_start']
            t_end = t_row['frame_end']
            
            p_starts = pred_group['frame_start'].values
            p_ends = pred_group['frame_end'].values
            
            overlap_starts = np.maximum(t_start, p_starts)
            overlap_ends = np.minimum(t_end, p_ends)
            overlap_durations = np.maximum(0, (overlap_ends - overlap_starts) + 1)
            
            valid_mask = overlap_durations > 0
            
            if not np.any(valid_mask):
                matches.append((hum_id, obj_id, t_idx, [], f"{t_start} - {t_end}", [], 0, (t_end - t_start) + 1, 0.0))
                continue
                
            v_p_starts = p_starts[valid_mask]
            v_p_ends = p_ends[valid_mask]
            v_overlap_durations = overlap_durations[valid_mask]
            
            intersection = np.sum(v_overlap_durations)
            
            t_duration = (t_end - t_start) + 1
            p_duration = np.sum((v_p_ends - v_p_starts) + 1)
            union = t_duration + p_duration - intersection
            
            tiou = intersection / union if union > 0 else 0
            
            t_frames = f"{t_start} - {t_end}"
            p_frames = [f"{s} - {e}" for s, e in zip(v_p_starts, v_p_ends)]
            p_indices = pred_group.index[valid_mask].tolist()
            
            matches.append((hum_id, obj_id, t_idx, p_indices, t_frames, p_frames, intersection, union, tiou))
            
    return pd.DataFrame(matches, columns=['human_id', 'object_id', 'truth_idx', 'pred_indices', 'truth_frames', 'pred_frames', 'intersection', 'union', 'tiou'])

In [ ]:
video_name = "vid21"
gt_csv_file = ground_truth_directory / (video_name + "_true_summary.csv")
pred_csv_file = predicted_directory / (video_name + "_pred_summary.csv")

result_df = evaluate_tiou(gt_csv_file, pred_csv_file)

print(result_df.drop(columns=['truth_idx', 'pred_indices']))
print(f"Mean TIOU: {result_df["tiou"].mean()}")

    human_id  object_id truth_frames                                                                     pred_frames  intersection  union      tiou
0          2          3      0 - 852                                                                       [0 - 615]           616    853  0.722157
1          2          3    847 - 921                                                                     [890 - 890]             1     75  0.013333
2          2         22   922 - 1038                                                                              []             0    117  0.000000
3          3          7    157 - 705                                                          [155 - 195, 545 - 545]            40    551  0.072595
4          3         14    706 - 736                                                                     [705 - 730]            25     32  0.781250
5         17         26   944 - 1090                                                                   [1030 - 1